# Carbon Arc Guide to Batching Data

This notebook explains and demonstrates a **small-batch processing pattern** for reading data from APIs and writing results to disk
without hitting rate limits or memory ceilings.  It will work best for users having trouble pulling large volumes of data out of Carbon Arc APIs.

**What the script does**
1. Uses a fixed **BATCH_SIZE** to fetch small chunks of records per request.
2. Adds a **page/offset** parameter so each request advances deterministically.
3. Implements **retry with exponential backoff** on transient errors (e.g., HTTP 429/5xx).
4. **Persists progress** after each batch so runs can resume if interrupted.
5. Streams results to **CSV/Parquet** incrementally to keep memory usage bounded.
6. Optionally **sleeps between batches** to be friendly to provider rate limits.
7. Provides simple **metrics logging**: items processed, batches, elapsed time.

Use this pattern when datasets are large, when provider limits are strict, or when you need robust, restartable jobs.

## Prerequisites
- Python 3.9+
- Carbon Arc Python SDK and pandas installed
- An API token available via environment variable `CARBONARC_TOKEN`

## Environment Setup

Install or upgrade the Carbon Arc Python SDK and common utils:

In [1]:
# If running locally, uncomment the pip install line:
# %pip install --upgrade carbonarc pandas matplotlib

import os
import pandas as pd

# Read API token from an environment variable for safety
TOKEN = os.getenv("CARBONARC_TOKEN", "<PUT_YOUR_TOKEN_HERE>")
assert TOKEN and TOKEN != "<PUT_YOUR_TOKEN_HERE>", "Please set CARBONARC_TOKEN or edit TOKEN in this cell."

AssertionError: Please set CARBONARC_TOKEN or edit TOKEN in this cell.

## Batch Configuration (edit these values as needed)

In [ ]:
# Tweak batch/runtime behavior here.
BATCH_SIZE = 500            # number of records per request
MAX_PAGES  = None           # set an int to cap pages; None processes all
SLEEP_SECS = 0.5            # pause between requests
MAX_RETRIES = 5             # retry attempts for 429/5xx
BACKOFF_BASE = 1.5          # exponential backoff multiplier

# Output configuration
OUTPUT_DIR = "outputs"
OUTPUT_CSV = "results.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Processing Pattern
**Fetch → Append → Persist Cursor → Sleep → Repeat**

Minimal pseudocode:

```python
cursor = load_cursor()  # e.g., last offset/page
while True:
    data = fetch_page(cursor, limit=BATCH_SIZE)
    if not data:
        break
    append_to_csv(data, OUTPUT_CSV)
    cursor = advance_cursor(cursor, len(data))
    save_cursor(cursor)
    time.sleep(SLEEP_SECS)

In [ ]:
pip install carbonarc python- pdotenvandas pyspark

In [ ]:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("MyApp").getOrCreate()

In [ ]:
import os
from dotenv import load_dotenv
from carbonarc import CarbonArcClient
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
load_dotenv()  
HOST = "https://api.carbonarc.co"
TOKEN = os.getenv("PROD_TOKEN")

client = CarbonArcClient(host=HOST, token=TOKEN)

## Example Download
The example below uses POS Convenience Store data across three Insights — POS Spend, POS Transactions, and POS Stores — over a specified time period. This example also uses the wildcard "*" to retrieve all entities for a given representation (e.g., all Product Brands).


In [ ]:
def make_monthly_ranges(start_str: str, end_str:str):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end = datetime.strptime(end_str, "%Y-%m-%d")
    current = start
    while current <= end:
        next_month_same_day = current + relativedelta(months=1)
        window_end = next_month_same_day.replace(day=1)
        if window_end > end:
            window_end = end
        yield current.strftime("%Y-%m-%d"), window_end.strftime("%Y-%m-%d")
        current = window_end + timedelta(days=1)

def make_weekly_ranges(start_str: str, end_str:str):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end = datetime.strptime(end_str, "%Y-%m-%d")
    current = start
    while current <= end:
        next_week_same_day = current + relativedelta(weeks=1)
        window_end = next_week_same_day - relativedelta(days=1)
        if window_end > end:
            window_end = end
        yield current.strftime("%Y-%m-%d"), window_end.strftime("%Y-%m-%d")
        current = window_end + timedelta(days=1)

data = list(make_monthly_ranges("2021-01-01","2025-07-21"))

list(make_monthly_ranges("2021-01-01","2025-07-21"))

errors = []
entity_ids = [
    #(124, 'pos_spend_normalized'),
    (385, 'pos_spend'),
    (386, 'pos_transactions'),
    (388, 'pos_stores')
]

location = ['us','state'] #us
temporal = ['day']#'week', 'month', 'quarter'

In [ ]:
dfs = {}


for insight_id, table_name in entity_ids:
    for loc in location:
        if loc == 'state':
            date_windows = list(make_weekly_ranges("2021-01-01", "2025-07-21"))
        else:
            date_windows = list(make_monthly_ranges("2021-01-01", "2025-07-21"))
        for start_date, end_date in date_windows:
            for time_res in temporal:
                print(f"Processing: insight_id={insight_id}, table_name={table_name}, location={loc}, time_resolution={time_res}, start_date={start_date}, end_date={end_date}")
                framework = {
                    "entities": {"carc_name": "*", "representation": "product"},
                    "insight": {"insight_id": insight_id},
                    "filters": {
                        "date_resolution": time_res,
                        "location_resolution": loc,
                        "date_range": {
                            "start_date": start_date,
                            "end_date": end_date
                        }
                    },
                    "aggregate": "sum"
                }
                try:
                    price = client.explorer.check_framework_price(framework)
                    print(price)
                    frameworks_id = client.explorer.buy_frameworks([framework])
                    print(frameworks_id)
                    df = pd.DataFrame(client.explorer.get_framework_data(framework_id=frameworks_id['frameworks'][0]))
                    df = pd.json_normalize(df['data'])
                    print(df)
                    df['time_series'] = time_res
                    key = f"{table_name}_{loc}_{time_res}"
                    
                    if df.empty:
                        errors.append({
                            "insight_id": insight_id,
                            "table_name": table_name,
                            "location": loc,
                            "time": time_res,
                            "start_date": start_date,
                            "end_date": end_date,
                            "error": "Empty DataFrame"
                        })
                        continue
                    dfs[key] = df
                except Exception as e:
                    print(e)
                    errors.append({
                        "insight_id": insight_id,
                        "table_name": table_name,
                        "location": loc,
                        "time": time_res,
                        "error": str(e)
                    })
                    continue 
                #     # Convert to Spark DataFrame
                #     spark_df = spark.createDataFrame(df)
                    
                #     # Define final table name
                #     final_table_name = f"bronze.trial_data.{table_name}_{loc}"
                #     if loc == 'us':
                #         mode = "overwrite"
                #     else:
                #         mode = "overwrite" if start_date == "2021-01-01" and end_date == "2021-02-01" else "append"
                    
                #     print(f"Writing to table: {final_table_name}, mode={mode}")
                #     spark_df.write.option('overwriteSchema', 'true').saveAsTable(final_table_name, mode=mode)
                # except Exception as e:
                #     errors.append({
                #         "insight_id": insight_id,
                #         "table_name": table_name,
                #         "location": loc,
                #         "time": time_res,
                #         "error": str(e)
                #     })
                #     continue

if len(errors)>0:
    print(errors)
else:
    print('no errors')
# # Read the table and make it unique
# for table_name in ["pos_spend_state", "pos_transactions_state", "pos_stores_state"]:
#     spark_df = spark.table(f"bronze.trial_data.{table_name}")
#     unique_spark_df = spark_df.dropDuplicates()
#     unique_spark_df.write.option('overwriteSchema', 'true').saveAsTable(f"bronze.trial_data.{table_name}", mode="overwrite")

# print('done')